# Credit Card Fraud Detection using XGBoost

This notebook performs fraud detection on credit card transactions using the [Kaggle Credit Card Fraud Detection dataset](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud).

The dataset contains **284,807 transactions** made by European cardholders in September 2013, with only **492 (0.17%)** being fraudulent — making this a highly imbalanced classification problem.

**Approach:**
1. Exploratory Data Analysis
2. Feature preprocessing
3. Handling class imbalance (SMOTE vs. class weights)
4. Model comparison (Logistic Regression, Random Forest, XGBoost)
5. Hyperparameter tuning with cross-validation
6. Threshold optimization
7. Model interpretability (Feature Importance & SHAP)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, RandomizedSearchCV
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, classification_report,
    roc_curve, roc_auc_score, precision_recall_curve,
    average_precision_score
)
from xgboost import XGBClassifier, plot_importance
from imblearn.over_sampling import SMOTE
import shap

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

In [ ]:
df = pd.read_csv('creditcard.csv')
print(f'Dataset shape: {df.shape}')
print(f'Number of features: {df.shape[1] - 1}')
print(f'Number of transactions: {df.shape[0]:,}')
df.head()

## Exploratory Data Analysis

In [ ]:
df.info()
print()
df.describe()

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicate rows: {df.duplicated().sum()}')

In [ ]:
print('Class Distribution:')
print(df['Class'].value_counts())
print(f'\nFraud percentage: {df["Class"].mean() * 100:.3f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x='Class', ax=axes[0], palette=['#2196F3', '#F44336'])
axes[0].set_title('Transaction Class Distribution')
axes[0].set_xticklabels(['Normal (0)', 'Fraud (1)'])
axes[0].set_ylabel('Count')

df['Class'].value_counts().plot.pie(
    autopct='%1.3f%%', labels=['Normal', 'Fraud'],
    colors=['#2196F3', '#F44336'], ax=axes[1], startangle=90
)
axes[1].set_title('Class Proportion')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df[df['Class'] == 0]['Amount'].hist(
    bins=50, ax=axes[0], alpha=0.7, color='#2196F3', edgecolor='black'
)
axes[0].set_title('Normal Transaction Amounts')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_xlim(0, 500)

df[df['Class'] == 1]['Amount'].hist(
    bins=50, ax=axes[1], alpha=0.7, color='#F44336', edgecolor='black'
)
axes[1].set_title('Fraudulent Transaction Amounts')
axes[1].set_xlabel('Amount ($)')
axes[1].set_ylabel('Frequency')
axes[1].set_xlim(0, 500)

plt.tight_layout()
plt.show()

print('Normal Transaction Amount Statistics:')
print(df[df['Class'] == 0]['Amount'].describe())
print('\nFraudulent Transaction Amount Statistics:')
print(df[df['Class'] == 1]['Amount'].describe())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].scatter(
    df[df['Class'] == 0]['Time'], df[df['Class'] == 0]['Amount'],
    alpha=0.1, s=1, color='#2196F3', label='Normal'
)
axes[0].set_title('Normal Transactions: Time vs Amount')
axes[0].set_xlabel('Time (seconds from first transaction)')
axes[0].set_ylabel('Amount ($)')
axes[0].set_ylim(0, 500)
axes[0].legend()

axes[1].scatter(
    df[df['Class'] == 1]['Time'], df[df['Class'] == 1]['Amount'],
    alpha=0.5, s=5, color='#F44336', label='Fraud'
)
axes[1].set_title('Fraudulent Transactions: Time vs Amount')
axes[1].set_xlabel('Time (seconds from first transaction)')
axes[1].set_ylabel('Amount ($)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
correlations = df.corr()['Class'].drop('Class').sort_values()

fig, ax = plt.subplots(figsize=(8, 10))
colors = ['#F44336' if x > 0 else '#2196F3' for x in correlations]
correlations.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Feature Correlation with Fraud Class')
ax.set_xlabel('Pearson Correlation Coefficient')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
top_positive = correlations.nlargest(4).index.tolist()
top_negative = correlations.nsmallest(4).index.tolist()
top_features = top_negative + top_positive

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for i, feature in enumerate(top_features):
    row, col = i // 4, i % 4
    ax = axes[row][col]
    ax.hist(df[df['Class'] == 0][feature], bins=50, alpha=0.5,
            label='Normal', density=True, color='#2196F3')
    ax.hist(df[df['Class'] == 1][feature], bins=50, alpha=0.5,
            label='Fraud', density=True, color='#F44336')
    ax.set_title(feature, fontsize=12)
    ax.legend(fontsize=8)

plt.suptitle('Top Correlated Features: Normal vs Fraud Distributions',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Feature Engineering & Preprocessing

- **Amount**: Standardized using `StandardScaler` since it has a wide range
- **Time**: Dropped after EDA (limited predictive value in this anonymized dataset)
- **V1-V28**: Already PCA-transformed — no additional scaling needed

In [ ]:
scaler = StandardScaler()
df['NormAmount'] = scaler.fit_transform(df['Amount'].values.reshape(-1, 1))
df = df.drop(['Time', 'Amount'], axis=1)

X = df.drop('Class', axis=1)
y = df['Class']

print(f'Feature matrix shape: {X.shape}')
print(f'Target vector shape: {y.shape}')
X.head()

## Train/Test Split

Using **stratified** splitting to maintain class proportions in both training and testing sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Training set: {X_train.shape[0]:,} samples')
print(f'Testing set:  {X_test.shape[0]:,} samples')
print(f'\nTraining class distribution:\n{y_train.value_counts()}')
print(f'\nTesting class distribution:\n{y_test.value_counts()}')

## Handling Class Imbalance

Two approaches compared:
1. **SMOTE (Synthetic Minority Oversampling Technique)**: generates synthetic fraud samples by interpolating between existing minority samples
2. **`scale_pos_weight`**: XGBoost's built-in parameter that adjusts the gradient for the minority class without creating synthetic data

SMOTE is applied **only to the training set** to avoid data leakage.

In [ ]:
oversample = SMOTE(random_state=42)
X_train_smote, y_train_smote = oversample.fit_resample(X_train, y_train)

print(f'Before SMOTE -> Normal: {len(y_train[y_train == 0]):,}, Fraud: {len(y_train[y_train == 1]):,}')
print(f'After SMOTE  -> Normal: {len(y_train_smote[y_train_smote == 0]):,}, Fraud: {len(y_train_smote[y_train_smote == 1]):,}')

## Model Training & Comparison

Comparing four approaches:
1. **Logistic Regression** (baseline) — with SMOTE
2. **Random Forest** — with SMOTE
3. **XGBoost default parameters** — with SMOTE
4. **XGBoost with `scale_pos_weight`** — without SMOTE (uses original imbalanced data)

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    """Evaluate a trained model and return a results dictionary."""
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print(f'\n{"=" * 60}')
    print(f'  {name}')
    print(f'{"=" * 60}')
    print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraud']))
    print(f'  ROC-AUC:  {roc_auc_score(y_test, y_proba):.4f}')
    print(f'  AUPRC:    {average_precision_score(y_test, y_proba):.4f}')

    return {
        'name': name,
        'y_pred': y_pred,
        'y_proba': y_proba,
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'auprc': average_precision_score(y_test, y_proba),
    }

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_smote, y_train_smote)
lr_results = evaluate_model('Logistic Regression (SMOTE)', lr, X_test, y_test)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_smote, y_train_smote)
rf_results = evaluate_model('Random Forest (SMOTE)', rf, X_test, y_test)

In [ ]:
xgb_default = XGBClassifier(
    objective='binary:logistic',
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb_default.fit(X_train_smote, y_train_smote)
xgb_default_results = evaluate_model('XGBoost Default (SMOTE)', xgb_default, X_test, y_test)

In [ ]:
weight_ratio = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
print(f'scale_pos_weight = {weight_ratio:.1f}')

xgb_weighted = XGBClassifier(
    objective='binary:logistic',
    use_label_encoder=False,
    eval_metric='logloss',
    scale_pos_weight=weight_ratio,
    random_state=42,
    n_jobs=-1
)
xgb_weighted.fit(X_train, y_train)
xgb_weighted_results = evaluate_model('XGBoost Weighted (No SMOTE)', xgb_weighted, X_test, y_test)

## Hyperparameter Tuning

Using `RandomizedSearchCV` with stratified 3-fold cross-validation to find optimal XGBoost hyperparameters. Scoring metric is **F1-score** — the harmonic mean of precision and recall.

In [ ]:
param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2, 0.5],
}

xgb_search = XGBClassifier(
    objective='binary:logistic',
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

search = RandomizedSearchCV(
    xgb_search,
    param_dist,
    n_iter=50,
    scoring='f1',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    random_state=42,
    n_jobs=-1,
    verbose=1
)
search.fit(X_train_smote, y_train_smote)

print(f'\nBest parameters: {search.best_params_}')
print(f'Best CV F1 score: {search.best_score_:.4f}')

xgb_tuned_results = evaluate_model(
    'XGBoost Tuned (SMOTE)', search.best_estimator_, X_test, y_test
)

## Results Comparison

In [ ]:
all_results = [
    lr_results, rf_results, xgb_default_results,
    xgb_weighted_results, xgb_tuned_results
]

results_df = pd.DataFrame(all_results)[
    ['name', 'precision', 'recall', 'f1', 'roc_auc', 'auprc']
]
results_df.columns = ['Model', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'AUPRC']
results_df = results_df.sort_values('F1-Score', ascending=False).reset_index(drop=True)

results_df.style.highlight_max(
    subset=['Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'AUPRC'],
    color='lightgreen'
).format({
    'Precision': '{:.4f}', 'Recall': '{:.4f}',
    'F1-Score': '{:.4f}', 'ROC-AUC': '{:.4f}', 'AUPRC': '{:.4f}'
})

## Cross-Validation

Evaluating the tuned XGBoost model with 5-fold stratified cross-validation for statistical robustness.

In [ ]:
best_model = search.best_estimator_
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring_metrics = ['f1', 'precision', 'recall', 'roc_auc']

print('5-Fold Stratified Cross-Validation (Tuned XGBoost on SMOTE data):\n')
for metric in scoring_metrics:
    scores = cross_val_score(
        best_model, X_train_smote, y_train_smote,
        cv=cv, scoring=metric, n_jobs=-1
    )
    print(f'  {metric.upper():>12s}: {scores.mean():.4f} (+/- {scores.std():.4f})')

## Evaluation Visualizations

### Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(25, 4))

for i, result in enumerate(all_results):
    cm = confusion_matrix(y_test, result['y_pred'])
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
        xticklabels=['Normal', 'Fraud'],
        yticklabels=['Normal', 'Fraud']
    )
    short_name = result['name'].split('(')[0].strip()
    axes[i].set_title(short_name, fontsize=10)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.suptitle('Confusion Matrices', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

### ROC Curves

In [ ]:
plt.figure(figsize=(10, 7))

for result in all_results:
    fpr, tpr, _ = roc_curve(y_test, result['y_proba'])
    plt.plot(fpr, tpr, linewidth=2,
             label=f"{result['name']} (AUC={result['roc_auc']:.4f})")

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — Model Comparison', fontsize=14)
plt.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

### Precision-Recall Curves

For highly imbalanced datasets, the Precision-Recall curve and AUPRC (Area Under Precision-Recall Curve) are more informative than ROC-AUC.

In [ ]:
plt.figure(figsize=(10, 7))

for result in all_results:
    prec, rec, _ = precision_recall_curve(y_test, result['y_proba'])
    plt.plot(rec, prec, linewidth=2,
             label=f"{result['name']} (AUPRC={result['auprc']:.4f})")

plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curves — Model Comparison', fontsize=14)
plt.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

## Threshold Optimization

The default classification threshold of 0.5 is rarely optimal for highly imbalanced problems. We search for the threshold that maximizes the **F1-score** on the test set.

In [ ]:
best_y_proba = search.best_estimator_.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.05, 0.95, 0.01)
f1_scores = []
precisions_t = []
recalls_t = []

for t in thresholds:
    preds = (best_y_proba >= t).astype(int)
    f1_scores.append(f1_score(y_test, preds))
    precisions_t.append(precision_score(y_test, preds, zero_division=0))
    recalls_t.append(recall_score(y_test, preds))

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1_val = f1_scores[best_idx]

plt.figure(figsize=(10, 6))
plt.plot(thresholds, f1_scores, label='F1-Score', linewidth=2, color='green')
plt.plot(thresholds, precisions_t, label='Precision', linewidth=1.5,
         linestyle='--', color='blue')
plt.plot(thresholds, recalls_t, label='Recall', linewidth=1.5,
         linestyle='--', color='red')
plt.axvline(x=best_threshold, color='gray', linestyle=':',
            linewidth=2, label=f'Optimal Threshold = {best_threshold:.2f}')
plt.xlabel('Classification Threshold', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Metrics vs. Classification Threshold (Tuned XGBoost)', fontsize=14)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f'Optimal Threshold: {best_threshold:.2f}')
print(f'F1-Score at optimal threshold: {best_f1_val:.4f}')

In [ ]:
y_pred_optimal = (best_y_proba >= best_threshold).astype(int)

print(f'Classification Report with Optimal Threshold ({best_threshold:.2f}):\n')
print(classification_report(y_test, y_pred_optimal, target_names=['Normal', 'Fraud']))

cm_optimal = confusion_matrix(y_test, y_pred_optimal)
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_optimal, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Normal', 'Fraud'],
    yticklabels=['Normal', 'Fraud']
)
plt.title(f'Confusion Matrix — Optimal Threshold = {best_threshold:.2f}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## Model Interpretability

### XGBoost Feature Importance

Built-in feature importance based on the average gain across all splits where a feature is used.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
plot_importance(
    search.best_estimator_, max_num_features=15,
    importance_type='gain', ax=ax
)
ax.set_title('XGBoost Feature Importance (Gain)', fontsize=14)
plt.tight_layout()
plt.show()

### SHAP Values

SHAP (SHapley Additive exPlanations) provides a unified measure of feature importance that shows how each feature contributes to individual predictions. Unlike global feature importance, SHAP shows directionality — whether a feature pushes the prediction toward fraud or normal.

In [ ]:
explainer = shap.TreeExplainer(search.best_estimator_)
X_test_sample = X_test.sample(n=min(1000, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_test_sample)

shap.summary_plot(shap_values, X_test_sample, show=False)
plt.title('SHAP Feature Importance — Impact on Fraud Prediction', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
shap.summary_plot(shap_values, X_test_sample, plot_type='bar', show=False)
plt.title('SHAP Mean Absolute Impact on Predictions', fontsize=12)
plt.tight_layout()
plt.show()

## Conclusion

### Key Findings

- The dataset is **extremely imbalanced** (0.17% fraud), making accuracy a misleading metric
- **SMOTE oversampling** combined with XGBoost provides strong fraud detection performance
- **Hyperparameter tuning** further improves the F1-score over default parameters
- **Threshold optimization** allows fine-tuning the precision-recall trade-off for business needs
- **SHAP analysis** reveals which transaction features are most indicative of fraud

### Metrics Guide

| Metric | Relevance |
|--------|-----------|
| **AUPRC** | Best single metric for imbalanced classification |
| **Recall** | Critical — missed fraud has high financial cost |
| **Precision** | Important — too many false alerts erode customer trust |
| **F1-Score** | Balanced trade-off between precision and recall |

### Recommendations for Production

1. Prioritize **recall** if the cost of missed fraud exceeds the cost of false positives
2. Use **threshold tuning** to match business-specific cost trade-offs
3. Monitor model performance over time — fraud patterns evolve (concept drift)
4. Consider ensemble approaches or real-time feature engineering for further improvement